In [ ]:

"""
CN7 10 - Test 원본 Subset / Scaling 복원 진단
==============================================

목적
----
원본 unlabeled_data.csv의 CN7 후보행과
최종 moldset_unlabeled_cn7.csv(35239행)의 관계를 복원한다.

핵심 아이디어
-------------
최종 test가 특정 raw subset을 StandardScaler한 것이라면,

    raw subset
        ↓ StandardScaler
    final standardized test

의 각 센서별 "정렬된 값(multiset)"이 정확히 일치해야 한다.

따라서 35,239 x 52,547 전체 거리행렬을 만들지 않는다.

진행
----
1) unlabeled_data.csv에서 CN7 후보 추출
2) metadata별 행 수 / 날짜 / serial 조사
3) metadata value 조합 중 정확히 35,239행이 되는 subset 후보 탐색
4) 각 후보를 StandardScaler한 뒤 최종 test와 센서별 sorted distribution 비교
5) exact 후보가 나오면 joint row multiset까지 검증
6) 가능하면 최종 test row 순서에 맞춰 raw metadata 복원

이 코드는 모델 학습을 하지 않으므로 비교적 빠르다.
"""

from pathlib import Path
from collections import Counter, defaultdict, deque
import itertools
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


# =============================================================================
# 0. 경로
# =============================================================================

BASE_DIR = Path(
    r"../../../data/raw"
)

CN7_WORK_DIR = Path(
    r"../../outputs/research"
)

SOURCE_PATH = (
    BASE_DIR
    / "unlabeled_data.csv"
)

TARGET_PATH = (
    BASE_DIR
    / "moldset_unlabeled_cn7.csv"
)

OUT_DIR = (
    CN7_WORK_DIR
    / "cn7_10_test_restore"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

EXPECTED_SOURCE_CN7 = 52547
EXPECTED_TARGET = 35239

SENSOR_COLS = [
    "Injection_Time",
    "Filling_Time",
    "Plasticizing_Time",
    "Cycle_Time",
    "Clamp_Close_Time",
    "Cushion_Position",
    "Plasticizing_Position",
    "Clamp_Open_Position",
    "Max_Injection_Speed",
    "Max_Screw_RPM",
    "Average_Screw_RPM",
    "Max_Injection_Pressure",
    "Max_Switch_Over_Pressure",
    "Max_Back_Pressure",
    "Average_Back_Pressure",
    "Barrel_Temperature_1",
    "Barrel_Temperature_2",
    "Barrel_Temperature_3",
    "Barrel_Temperature_4",
    "Barrel_Temperature_5",
    "Barrel_Temperature_6",
    "Hopper_Temperature",
    "Mold_Temperature_3",
    "Mold_Temperature_4",
]


# =============================================================================
# 유틸
# =============================================================================

def banner(title):
    print("\n" + "=" * 115)
    print(title)
    print("=" * 115)


def subset_sum_values(counts, target):
    """
    metadata value별 count를 이용해
    합이 target이 되는 value subset 하나를 찾는다.

    unique 수가 지나치게 많으면 사용하지 않는다.
    """
    items = list(counts.items())

    # DP: sum -> 선택된 value tuple
    dp = {
        0: ()
    }

    for value, count in items:
        current = list(
            dp.items()
        )

        for s, selected in current:
            ns = s + int(count)

            if ns > target:
                continue

            if ns not in dp:
                dp[ns] = (
                    selected
                    + (value,)
                )

            if ns == target:
                return dp[ns]

    return None


def standardize_candidate(df_candidate, sensor_cols):
    x = (
        df_candidate[
            sensor_cols
        ]
        .astype(float)
        .to_numpy()
    )

    mean = np.nanmean(
        x,
        axis=0,
    )

    std = np.nanstd(
        x,
        axis=0,
        ddof=0,
    )

    if np.any(
        std < 1e-15
    ):
        return None, mean, std

    z = (
        x - mean
    ) / std

    return z, mean, std


def sorted_distribution_error(
    candidate_df,
    target_df,
    sensor_cols,
):
    z, mean, std = standardize_candidate(
        candidate_df,
        sensor_cols,
    )

    if z is None:
        return {
            "mean_feature_mae":
                np.inf,
            "max_feature_mae":
                np.inf,
            "max_abs_error":
                np.inf,
        }

    t = (
        target_df[
            sensor_cols
        ]
        .astype(float)
        .to_numpy()
    )

    feature_maes = []
    max_errors = []

    for j in range(
        len(sensor_cols)
    ):
        a = np.sort(
            z[:, j]
        )

        b = np.sort(
            t[:, j]
        )

        diff = np.abs(
            a - b
        )

        feature_maes.append(
            float(
                np.mean(diff)
            )
        )

        max_errors.append(
            float(
                np.max(diff)
            )
        )

    return {
        "mean_feature_mae":
            float(
                np.mean(
                    feature_maes
                )
            ),
        "max_feature_mae":
            float(
                np.max(
                    feature_maes
                )
            ),
        "max_abs_error":
            float(
                np.max(
                    max_errors
                )
            ),
    }


def row_multiset_match(
    candidate_df,
    target_df,
    sensor_cols,
):
    z, mean, std = standardize_candidate(
        candidate_df,
        sensor_cols,
    )

    if z is None:
        return None

    t = (
        target_df[
            sensor_cols
        ]
        .astype(float)
        .to_numpy()
    )

    for decimals in [
        10,
        9,
        8,
        7,
        6,
    ]:
        zr = np.round(
            z,
            decimals,
        )

        tr = np.round(
            t,
            decimals,
        )

        c1 = Counter(
            map(
                tuple,
                zr,
            )
        )

        c2 = Counter(
            map(
                tuple,
                tr,
            )
        )

        if c1 == c2:
            return {
                "matched":
                    True,
                "decimals":
                    decimals,
                "z":
                    z,
                "mean":
                    mean,
                "std":
                    std,
            }

    return {
        "matched":
            False,
        "decimals":
            None,
        "z":
            z,
        "mean":
            mean,
        "std":
            std,
    }


# =============================================================================
# 1. 로드
# =============================================================================

banner(
    "CN7 10 - Test 원본 복원"
)

source = pd.read_csv(
    SOURCE_PATH
)

target = pd.read_csv(
    TARGET_PATH
)

print(
    "source full:",
    source.shape,
)

print(
    "target:",
    target.shape,
)

print(
    "target expected:",
    EXPECTED_TARGET,
)


# =============================================================================
# 2. CN7 원본 후보
# =============================================================================

banner(
    "1) Source CN7 후보"
)

if "PART_NAME" not in source.columns:
    raise KeyError(
        "PART_NAME 컬럼 없음"
    )

cn7_mask = (
    source[
        "PART_NAME"
    ]
    .astype(str)
    .str.contains(
        "CN7",
        case=False,
        na=False,
    )
)

source_cn7 = (
    source.loc[
        cn7_mask
    ]
    .copy()
)

source_cn7[
    "__source_row_index__"
] = source_cn7.index

source_cn7 = (
    source_cn7
    .reset_index(
        drop=True
    )
)

print(
    "source CN7:",
    source_cn7.shape,
)

if len(
    source_cn7
) != EXPECTED_SOURCE_CN7:
    print(
        f"[주의] 기존 예상 {EXPECTED_SOURCE_CN7}행과 다릅니다."
    )


# =============================================================================
# 3. 센서 확인
# =============================================================================

banner(
    "2) 공통 센서"
)

sensor_cols = [
    c
    for c in SENSOR_COLS
    if (
        c in source_cn7.columns
        and c in target.columns
    )
]

print(
    "공통 센서:",
    len(
        sensor_cols
    ),
)

print(
    sensor_cols
)

if len(
    sensor_cols
) != 24:
    print(
        "[주의] 24개가 아닙니다."
    )

print()

print(
    "Target standardized stats:"
)

stats = []

for c in sensor_cols:
    stats.append({
        "feature":
            c,
        "mean":
            float(
                pd.to_numeric(
                    target[c],
                    errors="coerce",
                ).mean()
            ),
        "std_ddof0":
            float(
                pd.to_numeric(
                    target[c],
                    errors="coerce",
                ).std(
                    ddof=0
                )
            ),
        "n_unique":
            target[c].nunique(
                dropna=False
            ),
    })

stats_df = pd.DataFrame(
    stats
)

print(
    stats_df
    .round(8)
    .to_string(
        index=False
    )
)

stats_df.to_csv(
    OUT_DIR
    / "target_scaled_stats.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 4. Metadata 후보
# =============================================================================

banner(
    "3) Metadata 구조"
)

metadata_cols = [
    c
    for c in source_cn7.columns
    if (
        c not in sensor_cols
        and c != "__source_row_index__"
    )
]

meta_profile = []

for c in metadata_cols:
    n_unique = source_cn7[
        c
    ].nunique(
        dropna=False
    )

    meta_profile.append({
        "column":
            c,
        "dtype":
            str(
                source_cn7[
                    c
                ].dtype
            ),
        "n_unique":
            n_unique,
        "missing":
            int(
                source_cn7[
                    c
                ]
                .isna()
                .sum()
            ),
    })

profile_df = (
    pd.DataFrame(
        meta_profile
    )
    .sort_values(
        "n_unique"
    )
)

print(
    profile_df
    .to_string(
        index=False
    )
)

profile_df.to_csv(
    OUT_DIR
    / "metadata_profile.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 5. 주요 metadata value counts
# =============================================================================

banner(
    "4) 주요 Metadata value counts"
)

candidate_meta_cols = []

for c in metadata_cols:
    n_unique = source_cn7[
        c
    ].nunique(
        dropna=False
    )

    if (
        2 <= n_unique <= 50
    ):
        candidate_meta_cols.append(
            c
        )

        print(
            f"\n[{c}] "
            f"unique={n_unique}"
        )

        print(
            source_cn7[
                c
            ]
            .value_counts(
                dropna=False
            )
            .head(60)
            .to_string()
        )


# =============================================================================
# 6. 정확히 35239행이 되는 metadata subset 탐색
# =============================================================================

banner(
    "5) Exact-count subset 후보 탐색"
)

candidate_defs = []

# -------------------------------------------------------------------------
# A. 단일 값
# -------------------------------------------------------------------------
for c in candidate_meta_cols:

    vc = source_cn7[
        c
    ].value_counts(
        dropna=False
    )

    for value, count in vc.items():

        if int(count) == EXPECTED_TARGET:

            candidate_defs.append({
                "type":
                    "single_value",
                "column":
                    c,
                "values":
                    (value,),
                "count":
                    int(count),
            })


# -------------------------------------------------------------------------
# B. 한 metadata column 내 여러 value 조합
# -------------------------------------------------------------------------
for c in candidate_meta_cols:

    vc = source_cn7[
        c
    ].value_counts(
        dropna=False
    )

    if len(vc) <= 30:

        solution = subset_sum_values(
            vc.to_dict(),
            EXPECTED_TARGET,
        )

        if (
            solution is not None
            and len(solution) > 0
        ):
            candidate_defs.append({
                "type":
                    "value_subset",
                "column":
                    c,
                "values":
                    tuple(
                        solution
                    ),
                "count":
                    EXPECTED_TARGET,
            })


# -------------------------------------------------------------------------
# C. 날짜 기준 cumulative 후보
# -------------------------------------------------------------------------
date_candidates = []

for c in metadata_cols:

    if (
        "DATE" in c.upper()
        or "TIME" in c.upper()
    ):
        parsed = pd.to_datetime(
            source_cn7[c],
            errors="coerce",
        )

        if (
            parsed.notna()
            .mean()
            >= 0.8
        ):
            dates = (
                parsed
                .dt.date
            )

            counts = (
                dates
                .value_counts()
                .sort_index()
            )

            # earliest cumulative
            cum = counts.cumsum()

            hit = cum[
                cum == EXPECTED_TARGET
            ]

            if len(hit):
                end_date = hit.index[0]

                candidate_defs.append({
                    "type":
                        "date_le",
                    "column":
                        c,
                    "values":
                        (end_date,),
                    "count":
                        EXPECTED_TARGET,
                })

            # latest cumulative
            rev = (
                counts
                .sort_index(
                    ascending=False
                )
                .cumsum()
            )

            hit = rev[
                rev == EXPECTED_TARGET
            ]

            if len(hit):
                start_date = hit.index[0]

                candidate_defs.append({
                    "type":
                        "date_ge",
                    "column":
                        c,
                    "values":
                        (start_date,),
                    "count":
                        EXPECTED_TARGET,
                })


# 중복 candidate 제거
seen = set()
unique_candidates = []

for d in candidate_defs:
    key = (
        d["type"],
        d["column"],
        tuple(
            map(
                str,
                d["values"],
            )
        ),
    )

    if key not in seen:
        seen.add(key)
        unique_candidates.append(
            d
        )

candidate_defs = unique_candidates

print(
    "exact-count candidate 수:",
    len(
        candidate_defs
    ),
)

for i, d in enumerate(
    candidate_defs,
    start=1,
):
    print(
        f"{i:02d}. "
        f"{d['type']} | "
        f"{d['column']} | "
        f"{d['values']}"
    )


# =============================================================================
# 7. 후보 실체화 + StandardScaler distribution exactness
# =============================================================================

banner(
    "6) Candidate scaling 검증"
)

candidate_results = []
candidate_frames = {}

for i, d in enumerate(
    candidate_defs,
    start=1,
):

    c = d[
        "column"
    ]

    typ = d[
        "type"
    ]

    values = d[
        "values"
    ]

    if typ in [
        "single_value",
        "value_subset",
    ]:

        mask = source_cn7[
            c
        ].isin(
            list(
                values
            )
        )

        # NaN value가 포함된 경우
        if any(
            pd.isna(v)
            for v in values
        ):
            mask = (
                mask
                |
                source_cn7[
                    c
                ].isna()
            )

    elif typ == "date_le":

        parsed = pd.to_datetime(
            source_cn7[c],
            errors="coerce",
        )

        mask = (
            parsed.dt.date
            <= values[0]
        )

    elif typ == "date_ge":

        parsed = pd.to_datetime(
            source_cn7[c],
            errors="coerce",
        )

        mask = (
            parsed.dt.date
            >= values[0]
        )

    else:
        continue

    cand = (
        source_cn7.loc[
            mask
        ]
        .copy()
    )

    if len(
        cand
    ) != EXPECTED_TARGET:
        continue

    err = sorted_distribution_error(
        cand,
        target,
        sensor_cols,
    )

    candidate_id = (
        f"cand_{i:03d}"
    )

    candidate_frames[
        candidate_id
    ] = cand

    row = {
        "candidate_id":
            candidate_id,
        "type":
            typ,
        "column":
            c,
        "values":
            " | ".join(
                map(
                    str,
                    values,
                )
            ),
        "rows":
            len(cand),
        **err,
    }

    candidate_results.append(
        row
    )

    print(
        f"{candidate_id} | "
        f"{typ:12s} | "
        f"{c:25s} | "
        f"MAE={err['mean_feature_mae']:.12g} | "
        f"MAX={err['max_abs_error']:.12g}"
    )

candidate_result_df = (
    pd.DataFrame(
        candidate_results
    )
)

if len(
    candidate_result_df
):

    candidate_result_df = (
        candidate_result_df
        .sort_values(
            [
                "mean_feature_mae",
                "max_abs_error",
            ]
        )
    )

    candidate_result_df.to_csv(
        OUT_DIR
        / "candidate_scaling_scores.csv",
        index=False,
        encoding="utf-8-sig",
    )

    print()

    print(
        "상위 후보:"
    )

    print(
        candidate_result_df
        .head(20)
        .to_string(
            index=False
        )
    )

else:

    print(
        "단순 metadata exact-count 후보가 없습니다."
    )


# =============================================================================
# 8. 최고 후보 joint row multiset 검증
# =============================================================================

banner(
    "7) Joint row multiset 검증"
)

best_candidate_id = None
match_result = None
best_candidate = None

if len(
    candidate_result_df
):

    for _, row in (
        candidate_result_df
        .head(10)
        .iterrows()
    ):

        candidate_id = row[
            "candidate_id"
        ]

        cand = candidate_frames[
            candidate_id
        ]

        check = row_multiset_match(
            cand,
            target,
            sensor_cols,
        )

        print(
            candidate_id,
            "matched=",
            check[
                "matched"
            ],
            "decimals=",
            check[
                "decimals"
            ],
        )

        if check[
            "matched"
        ]:

            best_candidate_id = (
                candidate_id
            )

            best_candidate = (
                cand.copy()
            )

            match_result = (
                check
            )

            break


# =============================================================================
# 9. Exact 복원 시 target row -> source row mapping
# =============================================================================

banner(
    "8) Target row 순서 복원"
)

restored = None

if (
    best_candidate_id
    is not None
):

    decimals = match_result[
        "decimals"
    ]

    z = match_result[
        "z"
    ]

    target_matrix = (
        target[
            sensor_cols
        ]
        .astype(float)
        .to_numpy()
    )

    zr = np.round(
        z,
        decimals,
    )

    tr = np.round(
        target_matrix,
        decimals,
    )

    buckets = defaultdict(
        deque
    )

    candidate_indices = (
        best_candidate.index
        .to_numpy()
    )

    for pos, key in enumerate(
        map(
            tuple,
            zr,
        )
    ):
        buckets[
            key
        ].append(
            pos
        )

    selected_positions = []

    failed = 0

    for key in map(
        tuple,
        tr,
    ):
        if (
            key not in buckets
            or len(
                buckets[key]
            ) == 0
        ):
            selected_positions.append(
                None
            )

            failed += 1

        else:

            selected_positions.append(
                buckets[
                    key
                ].popleft()
            )

    print(
        "mapping failed:",
        failed,
    )

    if failed == 0:

        restored = (
            best_candidate
            .iloc[
                selected_positions
            ]
            .copy()
            .reset_index(
                drop=True
            )
        )

        restored.insert(
            0,
            "target_row_index",
            np.arange(
                len(restored)
            ),
        )

        # scaled 원본도 함께 prefix 붙여 저장
        target_prefixed = (
            target[
                sensor_cols
            ]
            .copy()
        )

        target_prefixed.columns = [
            "scaled__"
            + c
            for c in sensor_cols
        ]

        restored = pd.concat(
            [
                restored,
                target_prefixed
                .reset_index(
                    drop=True
                ),
            ],
            axis=1,
        )

        restored.to_csv(
            OUT_DIR
            / "cn7_test_restored_35239.csv",
            index=False,
            encoding="utf-8-sig",
        )

        print(
            "복원 완료:",
            restored.shape,
        )


# =============================================================================
# 10. 복원 데이터 metadata 요약
# =============================================================================

banner(
    "9) 복원 Test metadata 요약"
)

summary_lines = []

if (
    restored is not None
):

    print(
        "best candidate:",
        best_candidate_id,
    )

    summary_lines.append(
        f"- exact candidate: {best_candidate_id}"
    )

    # Side
    if "PART_NAME" in restored.columns:

        s = (
            restored[
                "PART_NAME"
            ]
            .astype(str)
            .str.upper()
        )

        restored[
            "Side"
        ] = np.where(
            s.str.contains(
                r"\bRH\b",
                regex=True,
                na=False,
            ),
            "RH",
            np.where(
                s.str.contains(
                    r"\bLH\b",
                    regex=True,
                    na=False,
                ),
                "LH",
                "UNKNOWN",
            ),
        )

        print()

        print(
            "Side:"
        )

        print(
            restored[
                "Side"
            ]
            .value_counts(
                dropna=False
            )
            .to_string()
        )

    # TimeStamp
    timestamp_cols = [
        c
        for c in restored.columns
        if (
            "TIME" in c.upper()
            and not c.startswith(
                "scaled__"
            )
        )
    ]

    for c in timestamp_cols:

        parsed = pd.to_datetime(
            restored[c],
            errors="coerce",
        )

        if (
            parsed.notna()
            .mean()
            >= 0.8
        ):

            print()

            print(
                f"{c} range:"
            )

            print(
                parsed.min(),
                "~",
                parsed.max(),
            )

            summary_lines.append(
                f"- {c}: {parsed.min()} ~ {parsed.max()}"
            )

    # 주요 metadata
    for c in [
        "PART_FACT_PLAN_DATE",
        "PART_FACT_SERIAL",
        "EQUIP_NAME",
    ]:

        if c in restored.columns:

            print()

            print(
                f"{c}:"
            )

            print(
                restored[
                    c
                ]
                .value_counts(
                    dropna=False
                )
                .head(30)
                .to_string()
            )

else:

    summary_lines.append(
        "- 단순 metadata subset 방식으로 exact 복원되지 않음."
    )


# =============================================================================
# 11. 자동 요약
# =============================================================================

banner(
    "10) 자동 요약"
)

if len(
    candidate_result_df
):

    top = (
        candidate_result_df
        .iloc[0]
    )

    print(
        f"- best candidate score: "
        f"{top['candidate_id']} | "
        f"mean sorted MAE="
        f"{top['mean_feature_mae']:.12g}, "
        f"max error="
        f"{top['max_abs_error']:.12g}"
    )

    summary_lines.append(
        f"- best candidate sorted MAE: "
        f"{top['mean_feature_mae']:.12g}"
    )

if (
    restored is not None
):

    print(
        "- Test raw subset exact 복원 성공"
    )

    print(
        f"- 복원 행 수: "
        f"{len(restored)}"
    )

    print(
        "- cn7_test_restored_35239.csv 저장 완료"
    )

    summary_lines.append(
        "- Test raw subset exact 복원 성공"
    )

else:

    print(
        "- 아직 exact 복원되지 않았습니다."
    )

    print(
        "- 다음 단계에서 2개 metadata 조합 / 더 세밀한 subset 탐색이 필요합니다."
    )

with open(
    OUT_DIR
    / "test_restore_summary.txt",
    "w",
    encoding="utf-8",
) as f:

    f.write(
        "\n".join(
            summary_lines
        )
    )


# =============================================================================
# 완료
# =============================================================================

banner(
    "완료"
)

print(
    "결과 폴더:",
    OUT_DIR,
)

print()

print(
    "핵심 파일:"
)

print(
    "  1) metadata_profile.csv"
)

print(
    "  2) target_scaled_stats.csv"
)

print(
    "  3) candidate_scaling_scores.csv (후보가 있을 경우)"
)

print(
    "  4) cn7_test_restored_35239.csv (exact 성공 시)"
)

print(
    "  5) test_restore_summary.txt"
)

print()

print(
    "실행 후 5) Exact-count subset 후보 탐색부터 "
    "10) 자동 요약까지 보내주세요."
)
